# Phase 2 Worksheet — Chunking Strategies, Compared in Chroma
**Corrected in this version:** all embedding calls go through `embedder.embed_query()` / `embedder.embed_documents()` instead of `get_embedding(text, model=MODEL_JINA)`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

## Source document for this worksheet (deliberately structured)

In [ ]:
source_doc = """
# Customer Search API

## Authentication
Requires an API key sent in the Authorization header as a Bearer token.

## Request
GET /customers/search?query={text}&limit={n}

## Response
Returns a JSON array of customer objects matching the query.

## Error Codes
401: missing or invalid API key.
404: no customers matched the query.
500: internal server error, retry with backoff.
"""
print(source_doc)

## 1. Fixed-size chunking

In [ ]:
def fixed_size_chunk(text, size=120, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+size].strip())
        start += size - overlap
    return [c for c in chunks if c]

fixed_chunks = fixed_size_chunk(source_doc)
for c in fixed_chunks:
    print("-", repr(c[:60]))

## 2. Recursive chunking (heading > paragraph > sentence)

In [ ]:
import re

def recursive_chunk(text):
    sections = re.split(r"\n(?=## )", text.strip())
    return [s.strip() for s in sections if s.strip()]

recursive_chunks = recursive_chunk(source_doc)
for c in recursive_chunks:
    print("---")
    print(c[:100])

## 3. Semantic chunking (split where adjacent-sentence similarity drops)

In [ ]:
import numpy as np

def semantic_chunk(text, threshold=0.5):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.replace("\n", " ")) if s.strip()]
    if len(sentences) < 2:
        return sentences
    vecs = [np.array(v) for v in embedder.embed_documents(sentences)]
    chunks, current = [], [sentences[0]]
    for i in range(1, len(sentences)):
        sim = np.dot(vecs[i-1], vecs[i]) / (np.linalg.norm(vecs[i-1]) * np.linalg.norm(vecs[i]) + 1e-8)
        if sim < threshold:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])
    chunks.append(" ".join(current))
    return chunks

semantic_chunks = semantic_chunk(source_doc)
for c in semantic_chunks:
    print("---")
    print(c[:150])

## 4. Parent-child chunking, done CORRECTLY (this phase's teaser, solved)

In [ ]:
parent_id = "customer_search_api"
parent_text = source_doc

children = [
    ("auth", "Requires an API key sent in the Authorization header as a Bearer token."),
    ("request", "GET /customers/search?query={text}&limit={n}"),
    ("response", "Returns a JSON array of customer objects matching the query."),
    ("errors", "401: missing or invalid API key. 404: no customers matched. 500: internal server error."),
]

collection = client.get_or_create_collection("phase2_parent_child")

child_ids = [f"{parent_id}_{name}" for name, _ in children]
child_texts = [text for _, text in children]                      # <- EMBED THE CHILD, not the parent
child_embeddings = embedder.embed_documents(child_texts)
child_metadatas = [{"parent_id": parent_id, "parent_text": parent_text} for _ in children]

collection.upsert(ids=child_ids, embeddings=child_embeddings, documents=child_texts, metadatas=child_metadatas)

def parent_child_query(query_text, k=1):
    q_vec = embedder.embed_query(query_text)
    results = collection.query(query_embeddings=[q_vec], n_results=k)
    parent = results["metadatas"][0][0]["parent_text"]
    matched_child = results["documents"][0][0]
    return matched_child, parent

child_match, parent_returned = parent_child_query("what auth header format is required?")
print("Matched child chunk:", child_match)
print("\nReturned parent (full context):\n", parent_returned)

## 5. Table-aware chunking

In [ ]:
table_text = """
| id | service | status |
|----|---------|--------|
| 1  | Payment | Failed |
| 2  | Auth    | OK     |
| 3  | Payment | OK     |
"""

def table_aware_chunk(md_table):
    lines = [l for l in md_table.strip().splitlines() if l.strip().startswith("|")]
    header, _, *rows = lines
    return [f"{header}\n{row}" for row in rows]  # each chunk keeps the header WITH its row

for chunk in table_aware_chunk(table_text):
    print("---")
    print(chunk)

## 6. Hierarchical chunking

In [ ]:
hierarchy = {
    "book": "RAG Engineering Guide",
    "chapters": [
        {"title": "Chunking", "sections": [
            {"title": "Fixed-size", "text": "Splits by character count."},
            {"title": "Semantic", "text": "Splits by embedding similarity drop."},
        ]},
    ],
}

def flatten_hierarchical(hierarchy):
    chunks = []
    for chapter in hierarchy["chapters"]:
        chunks.append({"level": "chapter", "path": chapter["title"], "text": chapter["title"]})
        for section in chapter["sections"]:
            path = f"{chapter['title']} > {section['title']}"
            chunks.append({"level": "section", "path": path, "text": section["text"]})
    return chunks

for c in flatten_hierarchical(hierarchy):
    print(c["level"], "|", c["path"], "|", c["text"])

## 7. Sliding window chunking (for logs)

In [ ]:
log_lines = [f"2026-06-25 10:00:{i:02d} INFO request_id={i} status=200" for i in range(10)]

def sliding_window_chunk(lines, window=4, step=2):
    chunks = []
    for i in range(0, len(lines), step):
        chunks.append("\n".join(lines[i:i+window]))
    return chunks

for c in sliding_window_chunk(log_lines)[:3]:
    print("---")
    print(c)

## 8. Graph chunking

In [ ]:
import networkx as nx

g = nx.DiGraph()
g.add_edge("Service A", "Service B", relation="calls")
g.add_edge("Service B", "Service C", relation="depends_on_for_auth")

print("Nodes:", list(g.nodes))
print("Edges:", list(g.edges(data=True)))

def what_does_x_depend_on(graph, node):
    return list(graph.successors(node))

print("\nService A calls:", what_does_x_depend_on(g, "Service A"))
print("Which then depends on:", what_does_x_depend_on(g, "Service B"))

## Compare retrieval: same query, 3 chunking strategies, 3 collections

In [ ]:
strategies = {"fixed_size": fixed_chunks, "recursive": recursive_chunks, "semantic": semantic_chunks}
query = "what does the API return when no customers match?"
q_vec = embedder.embed_query(query)

for name, chunks in strategies.items():
    coll = client.get_or_create_collection(f"phase2_{name}")
    ids = [f"{name}_{i}" for i in range(len(chunks))]
    embeddings = embedder.embed_documents(chunks)
    coll.upsert(ids=ids, embeddings=embeddings, documents=chunks)
    result = coll.query(query_embeddings=[q_vec], n_results=1)
    print(f"{name:12s} top hit: {result['documents'][0][0][:80]}")

## Teaser exercise
Which strategy's top hit actually contains the 404 error code text cleanly, without irrelevant surrounding content? That's the practical difference these strategies make — not abstract, directly visible in your own query result above.